# Quantum RNG oparty na fluktuacjach prozni (vacuum-fluctuation QRNG)

## Punkt wyjscia projektu (P7)

> Vacuum-fluctuation QRNGs are arguably the easiest way to do a serious physics-heavy project without building custom hardware yourself. Australian National University publicly states that its service supplies real-time random numbers generated by measuring quantum vacuum fluctuations, and it documents an API that returns data in JSON. On the research side, vacuum-fluctuation QRNGs span balanced-homodyne designs around hundreds of Mbit/s and integrated devices at 100 Gbit/s, so there is both an accessible public data source and a deep modern literature behind it.

P7 wykorzystuje to zalozenie bezposrednio: budujemy walidowalny generator, ktory pobiera probki z publicznego API ANU i mapuje je do interfejsu projektowego (`random_uint64`, `random_floats`, `random_bits`, `random_bytes`).

## Dlaczego ten kierunek jest dobry do projektu

1. **Silna podstawa fizyczna:** zrodlo losowosci pochodzi z pomiaru fluktuacji prozni (a nie z deterministycznej dynamiki PRNG).
2. **Dostepnosc praktyczna:** istnieje publiczny endpoint JSON, wiec nie musimy budowac wlasnego ukladu detekcji homodynowej.
3. **Skalowalnosc literaturowa:** od konstrukcji rzedu setek Mbit/s po uklady zintegrowane raportowane na poziomie 100 Gbit/s.

## Uwagi implementacyjne

- Notebook domyslnie korzysta z historycznego endpointu ANU (`qrng.anu.edu.au/API/jsonI.php`), zgodnego z dokumentacja JSON.
- W dokumentacji ANU jest tez wskazanie, ze API zostalo przeniesione na nowa platforme (AWS). W konstruktorze generatora mozna wiec podmienic `api_url` i (opcjonalnie) przekazac `api_key`.
- Ze wzgledu na to, ze generator jest sieciowy, test wizualny uruchamiamy z mniejsza probka (`n_samples=10_000`), zeby uniknac bardzo dlugich serii zapytan.

## Plan walidacji

1. Pobranie danych QRNG i normalizacja do formatu generatora w projekcie.
2. Uruchomienie `validate_generator(...)` na probce liczb, bitow i floatow.
3. Uruchomienie `run_visual_tests(...)` (histogram, scatter, obraz szumu, random walk).

### Referencje

- ANU QRNG API documentation: https://qrng.anu.edu.au/contact/api-documentation/
- ANU QRNG homepage: https://qrng.anu.edu.au/
- Prace techniczne wskazywane przez ANU: Appl. Phys. Lett. 98, 231103 (2011) oraz Phys. Rev. Applied 3, 054004 (2015)


In [ ]:
from ipow.generators import ANUVacuumQRNG
from ipow.tests import ValidationConfig, validate_generator, run_visual_tests

import json

cfg = ValidationConfig(n_numbers=1000, n_bits=20000)



In [ ]:
# Jesli korzystasz z nowego endpointu AWS ANU, podmien api_url i ustaw api_key.
# Przykład:
# generator = ANUVacuumQRNG(api_url="https://api.quantumnumbers.anu.edu.au/", api_key="YOUR_KEY")

generator = ANUVacuumQRNG(
    api_url="https://qrng.anu.edu.au/API/jsonI.php",
    api_key=None,
    max_chunk=1024,
    timeout_sec=20.0,
)

results = validate_generator(generator, cfg)
print(json.dumps(results, indent=2))

# Dla API sieciowego zmniejszamy liczbe probek, aby ograniczyc liczbe zapytan HTTP.
run_visual_tests(generator, n_samples=10_000)

